[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/06_KMeans_Clustering_Wine.ipynb)

# K-Means Clustering — Discovering Wine Groups (Unsupervised)

**Problem type:** UNSUPERVISED clustering (no labels used to train)

---

## How K-Means Works

K-Means iteratively finds compact, spherical groups in your data:

1. **Initialise** — pick *k* centroids at random (or via k-means++ smart seeding).
2. **Assign** — label every data point with the index of its *nearest* centroid (Euclidean distance).
3. **Update** — move each centroid to the *mean* of all points assigned to it.
4. **Repeat** steps 2–3 until assignments stop changing (convergence).

> **Key insight:** the algorithm finds structure **entirely without labels**. That is what makes it *unsupervised*. Contrast this with Logistic Regression (notebook 02) which needed labelled training examples.

---

## Dataset: sklearn `load_wine`

* **178 wine samples** from three Italian cultivars.
* **13 chemical features** (alcohol, malic acid, ash, …).
* We will cluster the samples **without ever touching the `target` column**.
* Afterwards we compare clusters vs. true cultivar labels *only as a sanity check* — clustering itself used no labels.

---

## Notebook outline

| Step | Description |
|------|-------------|
| 1 | Setup & imports |
| 2 | Load data, set labels aside |
| 3 | Standardise features (essential for distance-based methods) |
| 4 | Choose *k*: elbow plot + silhouette scores |
| 5 | Fit K-Means (k=3) |
| 6 | Visualise clusters vs. true labels in PCA-2D space |
| 7 | Evaluate cluster quality (Adjusted Rand Index, contingency table) |
| 8 | Findings & Try-it-yourself exercises |


## 1 · Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend (safe for Colab and local)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score,
)

np.random.seed(42)
print('Libraries loaded successfully.')


## 2 · Load Data


In [ ]:
# Load Wine dataset as a DataFrame
wine = load_wine(as_frame=True)
X = wine.frame.drop(columns=['target'])   # features only
y_true = wine.target                        # cultivar labels — set aside!

print(f'Dataset shape : {X.shape}')
print(f'Features      : {list(X.columns)}')
print(f'True classes  : {sorted(y_true.unique())}  (0, 1, 2 = three cultivars)')
print(f'\nFirst 5 rows:')
X.head()


## 3 · Preprocessing — StandardScaler

K-Means assigns points to the **nearest centroid by Euclidean distance**. If one feature (e.g., *proline* ≈ 750) has a vastly larger scale than another (e.g., *nonflavanoid_phenols* ≈ 0.3), that single feature will dominate all distance calculations and the clusters will be meaningless.

**StandardScaler** transforms each feature to zero mean and unit variance — giving every feature an equal voice.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('After scaling:')
print(f'  Mean per feature (should be ~0): {X_scaled.mean(axis=0).round(6)}')
print(f'  Std  per feature (should be ~1): {X_scaled.std(axis=0).round(6)}')


## 4 · Choosing *k* — Elbow Plot & Silhouette Score

Two complementary heuristics:

* **Inertia (elbow method)** — total within-cluster sum-of-squares. Decreases as *k* grows; look for the 'elbow' where gains flatten.
* **Silhouette score** — measures how tightly packed each cluster is vs. how far it is from the next nearest cluster. Ranges −1 … +1; **higher is better**. A clear peak is a strong signal for the right *k*.


In [ ]:
k_range = range(1, 11)
inertias = []
silhouettes = []   # silhouette requires k >= 2

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    if k >= 2:
        silhouettes.append(silhouette_score(X_scaled, labels))
    else:
        silhouettes.append(np.nan)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=7)
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k=3')
axes[0].set_xlabel('Number of clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].set_xticks(list(k_range))

# Silhouette
k_sil = [k for k in k_range if k >= 2]
sil_vals = [s for s in silhouettes if not np.isnan(s)]
axes[1].plot(k_sil, sil_vals, 'gs-', linewidth=2, markersize=7)
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k=3')
axes[1].set_xlabel('Number of clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs k', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].set_xticks(k_sil)

plt.suptitle('Choosing the optimal k — both metrics point to k = 3',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/_elbow_silhouette.png', dpi=96, bbox_inches='tight')
plt.close()
print('Elbow + Silhouette plot saved.')

best_k_sil = k_sil[int(np.argmax(sil_vals))]
print(f'Best k by silhouette : {best_k_sil}')
print(f'Silhouette scores    : { {k: round(s,4) for k, s in zip(k_sil, sil_vals)} }')


## 5 · Fit K-Means (k = 3)


In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

print('K-Means fitted successfully.')
print(f'Inertia (WCSS) : {kmeans.inertia_:.2f}')
unique, counts = np.unique(cluster_labels, return_counts=True)
print('Cluster sizes  :', dict(zip(unique.tolist(), counts.tolist())))


## 6 · Visualise — PCA 2D Projection

We use **PCA** purely for visualisation — we reduce 13 features to 2 principal components so we can plot them. K-Means was run on the full 13-dimensional scaled data, not on PCA components.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_explained = pca.explained_variance_ratio_
print(f'PC1 explains {var_explained[0]*100:.1f}% variance')
print(f'PC2 explains {var_explained[1]*100:.1f}% variance')
print(f'Total          {sum(var_explained)*100:.1f}%')

# Centroid coordinates in PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)

# ── Side-by-side: cluster colours vs true cultivar colours ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cluster_colors = ['#e74c3c', '#2ecc71', '#3498db']
cultivar_colors = ['#e67e22', '#8e44ad', '#1abc9c']
cluster_markers = ['o', 's', '^']

# Left: K-Means clusters
for c in range(3):
    mask = cluster_labels == c
    axes[0].scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=cluster_colors[c], marker=cluster_markers[c],
        s=60, alpha=0.8, label=f'Cluster {c}'
    )
# Mark centroids
axes[0].scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='*', s=250, zorder=5, label='Centroids'
)
axes[0].set_xlabel(f'PC1 ({var_explained[0]*100:.1f}% var)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({var_explained[1]*100:.1f}% var)', fontsize=11)
axes[0].set_title('K-Means Clusters (no labels used in training)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# Right: True cultivars
cultivar_names = wine.target_names
for c in range(3):
    mask = y_true == c
    axes[1].scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=cultivar_colors[c], marker=cluster_markers[c],
        s=60, alpha=0.8, label=f'Cultivar {c}: {cultivar_names[c]}'
    )
axes[1].set_xlabel(f'PC1 ({var_explained[0]*100:.1f}% var)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({var_explained[1]*100:.1f}% var)', fontsize=11)
axes[1].set_title('True Cultivar Labels (for reference only)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('PCA 2D — K-Means clusters vs. true cultivars',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/_pca_scatter.png', dpi=96, bbox_inches='tight')
plt.close()
print('PCA scatter plots saved.')


## 7 · Evaluate Cluster Quality vs. True Labels

> **Reminder:** This evaluation is a *sanity check* only. We are asking: "Did the algorithm — with zero label information — happen to find the same groupings that the true cultivar labels encode?" A high score tells us the chemical signatures of the three cultivars are naturally separable.

**Adjusted Rand Index (ARI):** measures agreement between two label assignments, corrected for chance. ARI = 1 means perfect agreement; ARI = 0 means random.


In [ ]:
ari = adjusted_rand_score(y_true, cluster_labels)
print(f'Adjusted Rand Index (ARI) : {ari:.4f}')
print()

# Contingency table: rows = true cultivar, columns = predicted cluster
contingency = pd.crosstab(
    y_true.rename('True Cultivar'),
    pd.Series(cluster_labels, name='K-Means Cluster')
)
print('Contingency table (rows=true cultivar, cols=cluster):')
print(contingency.to_string())
print()
print('Interpretation:')
print('  Each cultivar should map predominantly to one cluster.')
print(f'  ARI ~ {ari:.2f} indicates strong agreement between clusters and true labels.')


## Findings

| Observation | Detail |
|-------------|--------|
| **Optimal k** | k = 3, confirmed by both the elbow (kink in inertia curve) and the silhouette score (peak at k = 3). |
| **Cluster–cultivar alignment** | The contingency table shows each K-Means cluster maps almost exclusively to one true cultivar. |
| **Adjusted Rand Index** | ARI ≈ 0.85 – 0.90, indicating near-perfect unsupervised recovery of the true groupings. |
| **Scaling was essential** | The 13 features span very different numeric ranges. Without StandardScaler the algorithm would be dominated by high-magnitude features (e.g., *proline*), producing much worse clusters. |
| **PCA projection** | In 2D the three clusters are visually well-separated, explaining >55% of total variance. |


## Try It Yourself

Run the cells below and observe what changes.


In [ ]:
# ── Experiment 1: cluster WITHOUT scaling and watch ARI collapse ──────────────
km_unscaled = KMeans(n_clusters=3, n_init=10, random_state=42)
labels_unscaled = km_unscaled.fit_predict(X)  # raw, unscaled X
ari_unscaled = adjusted_rand_score(y_true, labels_unscaled)
print(f'ARI WITHOUT scaling : {ari_unscaled:.4f}  (was {ari:.4f} with scaling)')
print('Notice how much worse clustering becomes without standardisation!\n')

# ── Experiment 2: try k=2 and k=4 ─────────────────────────────────────────────
for k in [2, 4]:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_k = km_k.fit_predict(X_scaled)
    sil_k = silhouette_score(X_scaled, labels_k)
    ari_k = adjusted_rand_score(y_true, labels_k)
    print(f'k={k}  ->  Silhouette={sil_k:.4f}  |  ARI={ari_k:.4f}')

sil_3 = silhouette_score(X_scaled, cluster_labels)
print(f'k=3  ->  Silhouette={sil_3:.4f}  |  ARI={ari:.4f}  <-- our chosen model')
print('\nHigher silhouette confirms k=3 is the best choice.')
